# News Agent — Agente de Noticias Financieras

Agente que recolecta, filtra y resume noticias financieras desde feeds RSS de medios internacionales y regionales.

| Versión | Descripción | Estado |
|---------|-------------|--------|
| 0 | Lector de feeds RSS: título, enlace y fecha | COMPLETADA |
| 1 | Filtrado por palabras clave y exportación a CSV | Pendiente |
| 2 | Almacenamiento en base de datos y deduplicación | Pendiente |
| 3 | Análisis de sentimiento con NLP | Pendiente |
| 4 | Agente con LLM: resumen y alertas automáticas | Pendiente |

In [1]:
print("Hello World")
name = input("Dime tu nombre")
print(name)

Hello World
Santiago


In [8]:
# Versión 0 — News Agent: lector de feeds RSS de medios financieros.
# Crea la estructura de carpetas del proyecto si aún no existe.
# - data/raw/       → feeds RSS descargados sin procesar
# - data/processed/ → noticias limpias y filtradas
# - logs/           → registros de ejecución
# - config/         → fuentes RSS y parámetros de configuración

from pathlib import Path
import os

base = Path(os.getcwd())
print(f"Creando carpetas en: {base}")

for carpeta in ["data/raw", "data/processed", "logs", "config"]:
    ruta = base / carpeta
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"✓ {ruta}")

#Aquí se crearon las carpetas donde el agente se pondrá a almacenar la información de noticias que vaya recolectando

Creando carpetas en: g:\Data Lake\Noticias
✓ g:\Data Lake\Noticias\data\raw
✓ g:\Data Lake\Noticias\data\processed
✓ g:\Data Lake\Noticias\logs
✓ g:\Data Lake\Noticias\config


In [9]:
#Ahora instalaremos Feedparser
import feedparser
print(f"feedparser {feedparser.__version__} importado correctamente.")

feedparser 6.0.12 importado correctamente.


In [10]:
# Fuentes RSS del agente — medios financieros por cobertura geográfica

FEEDS = [
    {"nombre": "Reuters Business",  "url": "https://news.google.com/rss/search?q=when:24h+allinurl:reuters.com&ceid=US:en&hl=en-US&gl=US"},    # Internacional
    {"nombre": "Yahoo Finance",     "url": "https://finance.yahoo.com/news/rssindex"},            # Internacional (EE.UU.)
    {"nombre": "Expansión México",  "url": "https://expansion.mx/rss"},                           # México
    {"nombre": "Investing España",        "url": "https://es.investing.com/rss/news.rss"}, # España
]

In [11]:
# Recorre cada fuente RSS y muestra las primeras 5 noticias

for feed in FEEDS:

    # Descarga y parsea el feed
    resultado = feedparser.parse(feed["url"])

    # Encabezado del medio
    print(f"\n{'='*60}")
    print(f"  {feed['nombre']}")
    print(f"{'='*60}")

    # Itera sobre las primeras 5 entradas
    for entrada in resultado.entries[:5]:
        titulo = entrada.get("title", "Sin título")
        enlace = entrada.get("link", "Sin enlace")
        fecha  = entrada.get("published", "Fecha no disponible")

        print(f"\nTítulo : {titulo}")
        print(f"Enlace : {enlace}")
        print(f"Fecha  : {fecha}")
        print("-" * 60)


  Reuters Business

Título : The paper trail linking a US fuel trader to a notorious Mexican cartel - Reuters
Enlace : https://news.google.com/rss/articles/CBMiuAFBVV95cUxOX1BmcklqSnh3R0dvVWtKRG5yR29id3lXN09FeXJhWlN3dl9tZkRyNW5hemltRTVBYmkwTmotTEZRTE14YmdWMUVCSVZ6cmpERFhrS0NpOEpqUU04aEJDd0NXcnE0LW92dWxDRWN4Tnk0S2VpVXZLblBXVXhHLW8wd2pIU0hTSlRYbHh5VFpLTV92dWdKOUNVZE9jSXFOYjV3Nm1SX2thaWdrWkNuOE9KYTVIeVJ3UG1H?oc=5
Fecha  : Thu, 28 May 2026 16:36:04 GMT
------------------------------------------------------------

Título : US Treasury rolls out Trump Accounts app nationwide - Reuters
Enlace : https://news.google.com/rss/articles/CBMipwFBVV95cUxPVzF0MW9ybndBRG41RVFUa2ItMVpFNFJ0MFhBSFB5dkkwdEpPZVlhUDBMX1pMcTJsX09zakdfVVhvTXRGc0hNRmZkUWNDWDQ1bXBpbWVTT0c0NzlaUXlBMm9ORExXaVF6OGtTdWZTVE01U0x5UnZVS0hRSmpFY295aDJXbTBCRklWZWN1RFh1VW9kVDduMmVTbVdCeVM4ZzhaR20zN3B6WQ?oc=5
Fecha  : Thu, 28 May 2026 17:40:16 GMT
------------------------------------------------------------

Título : S&P 500, Nasdaq gain 

In [12]:
import subprocess, sys

# Instala python-dotenv si no está disponible en el entorno
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

import os
from dotenv import load_dotenv

# Carga las variables definidas en el archivo .env al entorno del proceso
load_dotenv()

# Lee la API key de Anthropic desde las variables de entorno
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# Confirma que la clave se cargó, mostrando solo los primeros 8 caracteres
if ANTHROPIC_API_KEY:
    print(f"ANTHROPIC_API_KEY cargada correctamente: {ANTHROPIC_API_KEY[:8]}...")
else:
    print("⚠️  ANTHROPIC_API_KEY no encontrada. Verifica que exista en el archivo .env")

ANTHROPIC_API_KEY cargada correctamente: sk-ant-a...


In [13]:
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# --- Recolección de titulares ---
# Recorre todos los feeds y acumula hasta 20 títulos en total
titulos = []
for feed in FEEDS:
    resultado = feedparser.parse(feed["url"])
    for entrada in resultado.entries:
        if len(titulos) >= 20:
            break
        titulo = entrada.get("title", "").strip()
        if titulo:
            titulos.append(f"[{feed['nombre']}] {titulo}")
    if len(titulos) >= 20:
        break

# --- Construcción del prompt ---
noticias_texto = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titulos))

prompt = f"""Eres un analista financiero senior. A continuación tienes hasta 20 titulares recientes de medios financieros internacionales y regionales:

{noticias_texto}

Tu tarea:
1. Selecciona las 5 noticias más relevantes para un profesional de finanzas interesado en mercados globales, situación política, economía y mercados bursátiles.
2. Para cada noticia seleccionada escribe un resumen de exactamente 2 frases: la primera explica el hecho, la segunda su implicación financiera.
3. Si el titular está en inglés, redacta el resumen en inglés. Si está en español, en español.
4. Responde SOLO con las 5 noticias numeradas, sin introducción ni texto adicional.

Formato de respuesta:
1. [Titular]
   Resumen: [Frase 1. Frase 2.]
"""

# --- Llamada a Anthropic y presentación del resultado ---
print("Analizando titulares con Claude...\n")
print("=" * 60)
respuesta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    #estamos usando de momento el modelo más económico en tokens para las pruebas
    max_tokens=1024,
    #De momento ponemos un máximo en el  número de tokens porque solo se está enfocando en hacer un resumen de las noticias como si fuera encabezado
    messages=[{"role": "user", "content": prompt}]
)
print(respuesta.content[0].text)
print("=" * 60)

Analizando titulares con Claude...

1. US first-quarter GDP growth revised lower to 1.6% pace
   Resumen: The US economy showed weaker-than-expected growth in Q1 2024, with GDP revised downward to 1.6% annual pace. This slowdown signals reduced economic momentum and could influence Federal Reserve policy decisions regarding interest rate cuts.

2. US new home sales slump in April amid higher mortgage rates, prices
   Resumen: New home sales declined significantly in April as elevated mortgage rates and elevated prices deterred buyers from the housing market. The weakness in residential construction threatens to drag overall economic growth and could pressure related equity sectors.

3. S&P 500, Nasdaq gain on report of Middle East peace deal progress
   Resumen: US equity markets rallied following reports of progress toward a Middle East peace agreement, reducing geopolitical risk premiums. This development improves investor sentiment and could support sustained gains in technology and